In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5)) # single channel, not (0.5, 0.5, 0.5)
])

trainset = MNIST(root="./data", train=True, download=True, transform=transform)
testset = MNIST(root="./data", train=False, download=True, transform=transform)

100%|██████████████████████████████████████| 9.91M/9.91M [00:03<00:00, 2.98MB/s]
100%|██████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 88.8kB/s]
100%|███████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 899kB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 2.71MB/s]


In [18]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build CNN Model

In [28]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # in_channels=1 (grayscale)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
            
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128, 256), # 28 -> 14 -> 7 -> 3 (integer division)
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [29]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Train CNN

In [30]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.16196425441803455
epoch=2/10 & loss=0.04278299585295055
epoch=3/10 & loss=0.029474461115082538
epoch=4/10 & loss=0.022870883106304057
epoch=5/10 & loss=0.019277745981452404
epoch=6/10 & loss=0.014978616815155322
epoch=7/10 & loss=0.013763201763239287
epoch=8/10 & loss=0.011574473881151199
epoch=9/10 & loss=0.010216625778266054
epoch=10/10 & loss=0.010458795403233363


### Evaluate CNN

In [34]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

correct_labels = 0
total_labels = 0
all_preds, all_labels = [], []
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        all_preds.extend(predicted.tolist())
        all_labels.extend(labels.tolist())

print(f"accuracy = {correct_labels/total_labels * 100}")
print("precision: ", precision_score(all_labels, all_preds, average='macro'))
print("recall: ", recall_score(all_labels, all_preds, average='macro'))
print("confusion matrix: ", confusion_matrix(all_labels, all_preds))

accuracy = 99.33
precision:  0.9932167878564622
recall:  0.9932468699399882
confusion matrix:  [[ 978    0    0    0    0    0    1    0    0    1]
 [   0 1131    0    1    0    1    2    0    0    0]
 [   1    1 1026    0    0    0    2    2    0    0]
 [   0    0    1 1003    0    3    0    1    2    0]
 [   0    0    0    0  976    0    0    0    3    3]
 [   1    0    0    4    0  885    2    0    0    0]
 [   5    2    0    0    0    2  947    0    2    0]
 [   0    3    2    0    0    0    0 1020    0    3]
 [   0    0    2    0    0    0    1    0  970    1]
 [   1    0    0    0    4    3    0    0    4  997]]


### RNN model

In [37]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 10) # 10 classes, not 1

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [41]:
input_size = 28 # each row of the image = one timestep's features

rnn_model = RNN(input_size)
criterion = nn.CrossEntropyLoss() # multiclass, not BCELoss
optimizer = optim.Adam(rnn_model.parameters())

### Train RNN

In [42]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()

        images = images.squeeze(1)   # (batch, 1, 28, 28) -> (batch, 28, 28)

        outputs = rnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.808084706317133
epoch=2/10 & loss=0.34772666270481245
epoch=3/10 & loss=0.24355878986354704
epoch=4/10 & loss=0.19650891731018577
epoch=5/10 & loss=0.18045476143921552
epoch=6/10 & loss=0.15501795546995648
epoch=7/10 & loss=0.1429289936193271
epoch=8/10 & loss=0.13384613687835778
epoch=9/10 & loss=0.13248697616187716
epoch=10/10 & loss=0.12704328812960622


### Evaluate RNN

In [ ]:
correct_labels = 0
total_labels = 0
all_preds, all_labels = [], []
rnn_model.eval()

with torch.no_grad():
    for images, labels in testloader:
        images = images.squeeze(1)
        outputs = rnn.model(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        all_preds.extend(predicted.tolist())
        all_labels.extend(labels.tolist())

print("accuracy = ", correct_labels/total_labels * 100)
print("precision:", precision_score(all_labels/all_preds, avegrage="macro"))
print("recall:", recall_score(all_labels/all_preds, avegrage="macro"))
print("confusion matrix:", confusion_matrix(all))